In [26]:
# Import basic libraries
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download stopwords
nltk.download('stopwords')

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\errol\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [27]:
# Load dataset (CHANGE PATH if needed)
df = pd.read_csv(r"C:\Users\errol\Downloads\IMDB_Dataset.csv")

# Show first rows
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [28]:
# Shape of dataset
print("Shape:", df.shape)

# Class distribution
print("\nSentiment Count:\n", df['sentiment'].value_counts())

# Sample rows
df.sample(5)

Shape: (50000, 2)

Sentiment Count:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64


,review,sentiment
44229,Why do I give this 1974 porn movie 7 points? B...,positive
9952,While I have a great respect for Disney's anim...,positive
40790,This is the touching story of two families in ...,positive
44817,there's only so much that i can take of Filipi...,positive
35781,This movie is awesome for three main reasons. ...,positive


In [30]:
# Initialize stopwords and stemmer
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

# Define cleaning function
def preprocess_text(text):
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Tokenization
    words = text.split()
    
    # Remove stopwords + stemming
    words = [stemmer.stem(word) for word in words if word not in stop_words]
    
    return " ".join(words)

In [31]:
# Apply cleaning
df['clean_text'] = df['review'].apply(preprocess_text)

# Show results
df[['review', 'clean_text']].head()

,review,clean_text
0,One of the other reviewers has mentioned that ...,one review mention watch oz episod youll hook ...
1,A wonderful little production. <br /><br />The...,wonder littl product br br film techniqu unass...
2,I thought this was a wonderful way to spend ti...,thought wonder way spend time hot summer weeke...
3,Basically there's a family where a little boy ...,basic there famili littl boy jake think there ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visual stun film...


In [32]:
print("Original:\n", df['review'][0])
print("\nCleaned:\n", df['clean_text'][0])

Original:
 One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due

In [44]:
# Convert sentiment labels to numeric
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Define features and target
X = df['clean_text']
y = df['sentiment']

In [34]:
# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [35]:
# Initialize CountVectorizer
bow = CountVectorizer(max_features=5000)

# Fit and transform training data
X_train_bow = bow.fit_transform(X_train)

# Transform test data
X_test_bow = bow.transform(X_test)

In [36]:
# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000)

# Fit and transform training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform test data
X_test_tfidf = tfidf.transform(X_test)

In [37]:
# Train Logistic Regression on TF-IDF
lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

# Predictions
y_pred_lr = lr.predict(X_test_tfidf)

In [38]:
# Train Naive Bayes on BoW
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

# Predictions
y_pred_nb = nb.predict(X_test_bow)

In [39]:
# Train Decision Tree on BoW
dt = DecisionTreeClassifier()
dt.fit(X_train_bow, y_train)

# Predictions
y_pred_dt = dt.predict(X_test_bow)

In [40]:
# Function to evaluate models
def evaluate_model(y_test, y_pred, model_name):
    print(f"\n--- {model_name} ---")
    
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))

In [41]:
# Evaluate all models
evaluate_model(y_test, y_pred_lr, "Logistic Regression (TF-IDF)")
evaluate_model(y_test, y_pred_nb, "Naive Bayes (BoW)")
evaluate_model(y_test, y_pred_dt, "Decision Tree (BoW)")


--- Logistic Regression (TF-IDF) ---
Accuracy: 0.8846
Precision: 0.8739172281039461
Recall: 0.9009724151617384
F1 Score: 0.8872386163767833

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.87      0.88      4961
           1       0.87      0.90      0.89      5039

    accuracy                           0.88     10000
   macro avg       0.89      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000


--- Naive Bayes (BoW) ---
Accuracy: 0.8444
Precision: 0.8513213637280613
Recall: 0.8374677515380036
F1 Score: 0.8443377350940376

Classification Report:

              precision    recall  f1-score   support

           0       0.84      0.85      0.84      4961
           1       0.85      0.84      0.84      5039

    accuracy                           0.84     10000
   macro avg       0.84      0.84      0.84     10000
weighted avg       0.84      0.84      0.84     10000


--- Decision Tree (BoW) 

In [42]:
# Create comparison table
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_nb),
        f1_score(y_test, y_pred_dt)
    ]
})

# Display sorted results
results.sort_values(by="Accuracy", ascending=False)

,Model,Accuracy,F1 Score
0,Logistic Regression,0.8846,0.887239
1,Naive Bayes,0.8444,0.844338
2,Decision Tree,0.7217,0.720498


In [45]:
''' - TF-IDF performed better than Bag of Words for Logistic Regression.
- Naive Bayes performed well with BoW due to its probabilistic nature.
- Decision Tree showed lower performance due to overfitting.
- Preprocessing improved model accuracy significantly.
- Logistic Regression achieved the best overall performance. '''

' - TF-IDF performed better than Bag of Words for Logistic Regression.\n- Naive Bayes performed well with BoW due to its probabilistic nature.\n- Decision Tree showed lower performance due to overfitting.\n- Preprocessing improved model accuracy significantly.\n- Logistic Regression achieved the best overall performance. '

In [46]:
'''Logistic Regression with TF-IDF vectorization provided the best results among all models. Proper preprocessing and feature engineering played a key role in improving performance.'''

'Logistic Regression with TF-IDF vectorization provided the best results among all models. Proper preprocessing and feature engineering played a key role in improving performance.'